In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
data = pd.read_csv("healthcare_dataset.csv")

print("Rows and Columns:", data.shape)

data.head()

Rows and Columns: (55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  str    
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  str    
 3   Blood Type          55500 non-null  str    
 4   Medical Condition   55500 non-null  str    
 5   Date of Admission   55500 non-null  str    
 6   Doctor              55500 non-null  str    
 7   Hospital            55500 non-null  str    
 8   Insurance Provider  55500 non-null  str    
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  str    
 12  Discharge Date      55500 non-null  str    
 13  Medication          55500 non-null  str    
 14  Test Results        55500 non-null  str    
dtypes: float64(1), int64(2), str(12)
memory usage: 12.3 MB


In [4]:
data.isnull().sum()

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

In [5]:
print("Duplicate Records:", data.duplicated().sum())

Duplicate Records: 534


In [6]:
# Remove unwanted spaces

for column in data.columns:
    if data[column].dtype == "object":
        data[column] = data[column].str.strip()

In [7]:
# Standardize text format

data["Name"] = data["Name"].str.title()

data["Gender"] = data["Gender"].str.title()

data["Blood Type"] = data["Blood Type"].str.upper()

data["Medical Condition"] = data["Medical Condition"].str.title()

data["Admission Type"] = data["Admission Type"].str.title()

data["Medication"] = data["Medication"].str.title()

data["Insurance Provider"] = data["Insurance Provider"].str.title()

data["Test Results"] = data["Test Results"].str.title()

In [8]:
# Fill missing numerical values

num_cols = data.select_dtypes(include=["int64", "float64"]).columns

for column in num_cols:
    data[column] = data[column].fillna(data[column].median())


# Fill missing categorical values

cat_cols = data.select_dtypes(include="object").columns

for column in cat_cols:
    data[column] = data[column].fillna(data[column].mode()[0])

C:\Users\abhij\AppData\Local\Temp\ipykernel_1812\1951045080.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = data.select_dtypes(include="object").columns


In [9]:
# Convert date columns

data["Date of Admission"] = pd.to_datetime(data["Date of Admission"])

data["Discharge Date"] = pd.to_datetime(data["Discharge Date"])

In [10]:
# Calculate hospital stay

data["Hospital Stay"] = (
    data["Discharge Date"] -
    data["Date of Admission"]
).dt.days

data[["Hospital Stay"]].head()

,Hospital Stay
0,2
1,6
2,15
3,30
4,20


In [11]:
# Remove duplicate records

data = data.drop_duplicates()

data = data.reset_index(drop=True)

print("New Shape:", data.shape)

New Shape: (54966, 16)


In [12]:
# Billing amount should not be negative

data["Billing Amount"] = data["Billing Amount"].abs()

In [13]:
# Remove records with incorrect dates

data = data[data["Discharge Date"] >= data["Date of Admission"]]

data = data.reset_index(drop=True)

In [14]:
# Convert text into numbers

data["Gender Label"] = data["Gender"].map({
    "Male": 0,
    "Female": 1
})

data["Blood Type Label"] = data["Blood Type"].map({
    "A+":0,
    "A-":1,
    "B+":2,
    "B-":3,
    "AB+":4,
    "AB-":5,
    "O+":6,
    "O-":7
})

data["Admission Type Label"] = data["Admission Type"].map({
    "Elective":0,
    "Urgent":1,
    "Emergency":2
})

data["Medical Condition Label"] = data["Medical Condition"].map({
    "Cancer":0,
    "Obesity":1,
    "Diabetes":2,
    "Asthma":3,
    "Hypertension":4,
    "Arthritis":5
})

data["Test Results Label"] = data["Test Results"].map({
    "Normal":0,
    "Abnormal":1,
    "Inconclusive":2
})

In [15]:
# Check the cleaned dataset

data.info()

data.head()

<class 'pandas.DataFrame'>
RangeIndex: 54966 entries, 0 to 54965
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Name                     54966 non-null  str           
 1   Age                      54966 non-null  int64         
 2   Gender                   54966 non-null  str           
 3   Blood Type               54966 non-null  str           
 4   Medical Condition        54966 non-null  str           
 5   Date of Admission        54966 non-null  datetime64[us]
 6   Doctor                   54966 non-null  str           
 7   Hospital                 54966 non-null  str           
 8   Insurance Provider       54966 non-null  str           
 9   Billing Amount           54966 non-null  float64       
 10  Room Number              54966 non-null  int64         
 11  Admission Type           54966 non-null  str           
 12  Discharge Date           54966 non-null  da

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,Hospital Stay,Gender Label,Blood Type Label,Admission Type Label,Medical Condition Label,Test Results Label
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,2,0,3,1,0,0
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6,0,0,2,1,2
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,15,1,1,2,1,0
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,30,1,6,0,2,1
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,20,1,4,1,0,1


In [16]:
# Save the cleaned dataset

data.to_csv("healthcare_dataset_cleaned_labelled.csv", index=False)

print("Dataset saved successfully.")

Dataset saved successfully.
